In [1]:
import sys
import pandas as pd

print(sys.executable)
print("pandas:", pd.__version__)
print("Ready!")


/Users/skymoses/Desktop/CDS3004_Project/.venv/bin/python
pandas: 2.3.3
Ready!


In [2]:
from pathlib import Path

print("Exploring the Link Between Study Habits and Academic Performance Among University Students (Responses).xlsx：", Path.cwd())

for file in Path(".").glob("*.xlsx"):
    print(file.name)

Exploring the Link Between Study Habits and Academic Performance Among University Students (Responses).xlsx： /Users/skymoses/Desktop/CDS3004_Project
survey_cleaned.xlsx
Exploring the Link Between Study Habits and Academic Performance Among University Students (Responses).xlsx


In [3]:
# Load the original survey data
original_file = "Exploring the Link Between Study Habits and Academic Performance Among University Students (Responses).xlsx"

raw = pd.read_excel(original_file, sheet_name="Form responses 1")

# Check the dataset size
print("Number of responses:", raw.shape[0])
print("Number of columns:", raw.shape[1])

# Preview the first five responses
display(raw.head())

Number of responses: 46
Number of columns: 12


,Timestamp,What is your current study year,Which Faculty do you studying,the range your Gpa grade of last semester,How much time you spent on studying per week ( outside of class time),How would you describe your class attendance last year,How much time do you usually spend on revision before exams or major assessments,Would you take notes during the lesson time/revision,How do you usually take a note,"How often do phone distraction(Notification, Social media, Message, etc.) interrupt your study session",What do you usually do when you got distracted by your phone ( you can choose more than one answer,Where you usually Study
0,2026-11-11 00:10:24.395,Year 5 or above,Medicine,3.5 above,21 hours above,Never Absent(100%),I start revision before 2 weeks of the exams o...,Yes（please answer next question),Both Handwritten and Digital,Every Study session,Continue using it for a while,Home
1,2026-11-11 00:11:12.720,Year 4,Engineering,below 2.5,21 hours above,Absent 1-2 classes,a few days before,Yes（please answer next question),Digital notes( lat top/ tablet,Almost never,Use app that block other distraction(eg. Fores...,Café/ Public place
2,2026-11-11 00:22:59.944,Year 2,Social Science,2.5-2.99,0-5 hours,Absent 3-4 classes,1-2 weeks before,Yes（please answer next question),Digital notes( lat top/ tablet,Sometimes,Switch on don't disturb mode or leave the phon...,Home
3,2026-11-11 00:23:56.251,Year 2,Arts,3.0-3.49,Seldom study beside the Final Exam Period,Absent 5-6 classes,last minutes/ Deadline Fighters,No (skip the next question),NaN,Often,Continue using it for a while,Home
4,2026-11-11 00:24:18.958,Year 4,Social Science,below 2.5,6-10 hours,Absent 5-6 classes,1-2 weeks before,Yes（please answer next question),Digital notes( lat top/ tablet,Often,"Check it quickly and put it away, Continue usi...",Dorm/ Hostel


In [4]:
# Check missing values, including empty or whitespace-only answers
missing_check = raw.replace(r"^\s*$", pd.NA, regex=True)

missing_summary = pd.DataFrame({
    "Missing count": missing_check.isna().sum(),
    "Missing percentage": (
        missing_check.isna().mean() * 100
    ).round(1)
})

display(missing_summary)

,Missing count,Missing percentage
Timestamp,0,0.0
What is your current study year,0,0.0
Which Faculty do you studying,0,0.0
the range your Gpa grade of last semester,0,0.0
How much time you spent on studying per week ( outside of class time),0,0.0
How would you describe your class attendance last year,0,0.0
How much time do you usually spend on revision before exams or major assessments,0,0.0
Would you take notes during the lesson time/revision,0,0.0
How do you usually take a note,10,21.7
"How often do phone distraction(Notification, Social media, Message, etc.) interrupt your study session",0,0.0


In [5]:
# Identify the note-taking questions
takes_notes = raw.iloc[:, 7].str.strip()
note_method = missing_check.iloc[:, 8]

# Check whether skipped answers follow the survey instructions
note_check = pd.crosstab(
    takes_notes,
    note_method.isna(),
    rownames=["Takes notes"],
    colnames=["Note method is missing"]
)

display(note_check)

Note method is missing,False,True
Takes notes,,
No (skip the next question),3,10
Yes（please answer next question),33,0


In [6]:
# Create a working copy and preserve the original data
clean = raw.copy()

# Keep the original note-taking method in a new column
clean["note_method_clean"] = note_method.copy()

# Mark the method as not applicable for respondents who answered No
no_notes = takes_notes.str.startswith("No", na=False)
clean.loc[no_notes, "note_method_clean"] = "Not applicable"

# Check the cleaned categories
display(
    clean["note_method_clean"]
    .value_counts(dropna=False)
    .rename_axis("Note-taking method")
    .to_frame("Count")
)

print("Number of responses retained:", len(clean))

,Count
Note-taking method,
Digital notes( lat top/ tablet,17
Not applicable,13
Both Handwritten and Digital,10
Handwritten notes,6


Number of responses retained: 46


In [7]:
# Count exact duplicate rows, including the timestamp
print("Exact duplicate rows:", raw.duplicated().sum())

# Find matching answers, excluding the timestamp
answer_columns = raw.columns[1:]
duplicate_answers = raw.duplicated(
    subset=answer_columns,
    keep=False
)

print("Responses with matching answers:", duplicate_answers.sum())

# Show their original Excel row numbers and submission times
duplicate_review = raw.loc[duplicate_answers].copy()
duplicate_review.insert(
    0, "Excel row", duplicate_review.index + 2
)

display(duplicate_review[["Excel row", "Timestamp"]])

Exact duplicate rows: 0
Responses with matching answers: 2


,Excel row,Timestamp
17,19,2026-11-11 19:04:04.763
45,47,2026-11-14 03:28:36.036


In [8]:
# Identify the faculty column
faculty_col = raw.columns[2]

# Review all faculty responses and their frequencies
display(
    raw[faculty_col]
    .value_counts(dropna=False)
    .rename_axis("Faculty response")
    .to_frame("Count")
)

,Count
Faculty response,
Social Science,17
Business,9
Engineering,6
Arts,6
Computer Science,3
Medicine,2
Music,1
Data Science,1
Associate of arts Gpa of last semester: 4.00,1


In [9]:
# Identify the ambiguous faculty response
ambiguous_faculty = (
    raw[faculty_col].str.strip()
    == "Associate of arts Gpa of last semester: 4.00"
)

# Apply the documented cleaning decision to the working copy
clean.loc[ambiguous_faculty, faculty_col] = "Unclassified"

# Verify the change
print("Faculty responses reclassified:", ambiguous_faculty.sum())

display(pd.DataFrame({
    "Original faculty": raw.loc[ambiguous_faculty, faculty_col],
    "Cleaned faculty": clean.loc[ambiguous_faculty, faculty_col]
}))

print("Number of responses retained:", len(clean))

Faculty responses reclassified: 1


,Original faculty,Cleaned faculty
40,Associate of arts Gpa of last semester: 4.00,Unclassified


Number of responses retained: 46


In [10]:
# Identify the phone-distraction response column
phone_response_col = raw.columns[10]

# Print each distinct response in full
phone_counts = raw[phone_response_col].value_counts(dropna=False)

for response, count in phone_counts.items():
    print(f"Count: {count}")
    print(response)
    print()

Count: 8
Continue using it for a while

Count: 8
Switch on don't disturb mode or leave the phone alone

Count: 8
Check it quickly and put it away

Count: 6
Switch on don't disturb mode or leave the phone alone, Check it quickly and put it away

Count: 5
Check it quickly and put it away, Continue using it for a while

Count: 3
Switch on don't disturb mode or leave the phone alone, Check it quickly and put it away, Use app that block other distraction(eg. Forest, Focus mode, etc.)

Count: 2
Use app that block other distraction(eg. Forest, Focus mode, etc.)

Count: 2
Switch on don't disturb mode or leave the phone alone, Check it quickly and put it away, Continue using it for a while

Count: 1
Check it quickly and put it away, Use app that block other distraction(eg. Forest, Focus mode, etc.)

Count: 1
I don’t really distract by phone cause I always turn it over 

Count: 1
Check it quickly and put it away, Use app that block other distraction(eg. Forest, Focus mode, etc.), Continue using 

In [11]:
# Identify the response containing unrelated revision information
mixed_response = (
    "Check it quickly and put it away, "
    "Start revision before 2 months of the exams or major assessments technically"
)

phone_issue = raw[phone_response_col].eq(mixed_response)

# Keep the valid phone-response text
clean.loc[phone_issue, phone_response_col] = (
    "Check it quickly and put it away"
)

# Verify the change
print("Phone responses corrected:", phone_issue.sum())

for index in raw.index[phone_issue]:
    print("Excel row:", index + 2)
    print("Before:", raw.loc[index, phone_response_col])
    print("After:", clean.loc[index, phone_response_col])

Phone responses corrected: 1
Excel row: 42
Before: Check it quickly and put it away, Start revision before 2 months of the exams or major assessments technically
After: Check it quickly and put it away


In [12]:
# Identify the study-location column
location_col = raw.columns[11]

# Review each location and its frequency
location_counts = raw[location_col].value_counts(dropna=False)

for location, count in location_counts.items():
    print(f"Count: {count}")
    print(location)
    print()

Count: 27
Home

Count: 9
Library

Count: 6
Dorm/ Hostel

Count: 1
Café/ Public place

Count: 1
School

Count: 1
random spot (no fixed sport every time) in campus that is quite enough to focus

Count: 1
HKCC Empty classrooms, empty rooms in library



In [13]:
# Define the three responses to group together
campus_locations = [
    "School",
    "random spot (no fixed sport every time) in campus that is quite enough to focus",
    "HKCC Empty classrooms, empty rooms in library"
]

# Create a cleaned location column
clean["study_location_clean"] = raw[location_col].replace({
    location: "Other campus locations"
    for location in campus_locations
})

# Verify the cleaned categories
display(
    clean["study_location_clean"]
    .value_counts(dropna=False)
    .rename_axis("Study location")
    .to_frame("Count")
)

print("Number of responses retained:", len(clean))

,Count
Study location,
Home,27
Library,9
Dorm/ Hostel,6
Other campus locations,3
Café/ Public place,1


Number of responses retained: 46


In [14]:
# Read the 46 survey responses, excluding the summary rows below
excel_clean = pd.read_excel(
    "survey_cleaned.xlsx",
    sheet_name="Form responses 1",
    nrows=46
)

print("Python dataset shape:", clean.shape)
print("Excel dataset shape:", excel_clean.shape)

# Compare column names, row order and values
try:
    pd.testing.assert_frame_equal(
        clean.reset_index(drop=True),
        excel_clean.reset_index(drop=True),
        check_dtype=False,
        check_exact=True
    )
    print("PASS: Python results match the manually cleaned Excel data.")
except AssertionError as error:
    print("Differences found:")
    print(error)

Python dataset shape: (46, 14)
Excel dataset shape: (46, 14)
PASS: Python results match the manually cleaned Excel data.


In [15]:
# Document the cleaning decisions and review findings
cleaning_log = pd.DataFrame([
    {
        "Field": "Note-taking method",
        "Finding": "10 instructed skips; 3 No responses with a method entered.",
        "Action": "Set note_method_clean to Not applicable for all 13 No responses."
    },
    {
        "Field": "Faculty",
        "Finding": "1 response mixed a programme name with GPA information.",
        "Action": "Label as Unclassified because the faculty could not be confirmed."
    },
    {
        "Field": "Phone response",
        "Finding": "1 response contained unrelated revision information.",
        "Action": "Remove the unrelated text and retain the valid phone response."
    },
    {
        "Field": "Study location",
        "Finding": "3 responses described campus locations in different ways.",
        "Action": "Group them as Other campus locations in study_location_clean."
    },
    {
        "Field": "Duplicate review",
        "Finding": "0 exact duplicates; 2 responses had identical answers but different timestamps.",
        "Action": "Retain both pending verification; matching answers alone do not prove duplicate submission."
    },
    {
        "Field": "Excel comparison",
        "Finding": "Python and manually cleaned Excel values matched across 46 rows and 14 columns.",
        "Action": "Record successful reproduction of the manual cleaning."
    }
])

display(cleaning_log)

,Field,Finding,Action
0,Note-taking method,10 instructed skips; 3 No responses with a met...,Set note_method_clean to Not applicable for al...
1,Faculty,1 response mixed a programme name with GPA inf...,Label as Unclassified because the faculty coul...
2,Phone response,1 response contained unrelated revision inform...,Remove the unrelated text and retain the valid...
3,Study location,3 responses described campus locations in diff...,Group them as Other campus locations in study_...
4,Duplicate review,0 exact duplicates; 2 responses had identical ...,Retain both pending verification; matching ans...
5,Excel comparison,Python and manually cleaned Excel values match...,Record successful reproduction of the manual c...


In [16]:
# Review the categories used in the analysis
review_columns = {
    "Study year": clean.columns[1],
    "GPA range": clean.columns[3],
    "Weekly study time": clean.columns[4],
    "Class attendance": clean.columns[5],
    "Revision timing": clean.columns[6],
    "Phone distraction frequency": clean.columns[9],
}

for label, column in review_columns.items():
    print(f"\n{label}")
    print(clean[column].value_counts(dropna=False).to_string())

# Count extra spaces at the start or end of text
header_spaces = sum(name != name.strip() for name in clean.columns)

cell_spaces = sum(
    isinstance(value, str) and value != value.strip()
    for column in clean.columns
    for value in clean[column]
)

print("\nColumn names with extra spaces:", header_spaces)
print("Text cells with extra spaces:", cell_spaces)


Study year
What is your current study year
Year 4             14
Year 2             14
Year 3             10
Year 5 or above     5
Year 1              3

GPA range
the range your Gpa grade of last semester 
3.0-3.49     21
3.5 above    17
2.5-2.99      5
below 2.5     3

Weekly study time
How much time you spent on studying per week ( outside of class time)
0-5 hours                                    12
6-10 hours                                   10
Seldom study beside the Final Exam Period     8
16-20 hours                                   6
21 hours above                                5
11-15 hours                                   5

Class attendance
How would you describe your class attendance last year
Never Absent(100%)    13
Absent 1-2 classes    13
Absent 3-4 classes    12
Absent 5-6 classes     8

Revision timing
How much time do you usually spend on revision before exams or major assessments
1-2 weeks before                                                     23
a few da

In [17]:
# Remove leading and trailing spaces from the export copy
final_data = clean.copy()
final_data.columns = final_data.columns.str.strip()

for column in final_data.columns:
    final_data[column] = final_data[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Record the final formatting cleanup
whitespace_log = pd.DataFrame([{
    "Field": "Whitespace",
    "Finding": f"{header_spaces} column names and {cell_spaces} text cells had extra spaces.",
    "Action": "Trimmed leading and trailing spaces in the exported dataset."
}])

final_log = pd.concat([cleaning_log, whitespace_log], ignore_index=True)

# Export the dataset and cleaning log to a separate workbook
output_file = "survey_cleaned_python.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_data.to_excel(writer, sheet_name="Cleaned Survey", index=False)
    final_log.to_excel(writer, sheet_name="Cleaning Log", index=False)

print("Saved:", output_file)
print("Dataset shape:", final_data.shape)
print("Cleaning log entries:", len(final_log))

Saved: survey_cleaned_python.xlsx
Dataset shape: (46, 14)
Cleaning log entries: 7
